# Step through a model with the Saturn debugger

Run all cells on CPU, without downloads or a scheduler. These are real native Qwen2 layers with random seed-7 weights; token IDs demonstrate mechanics, not trained language.

Install from the checkout: `pip install -e '.[ar,notebooks]'`, then `jupyter lab notebooks/01_debugger.ipynb`.

Each cell is a stop you can inspect and change. The notebook uses the same `Debugger.execute` command parser as `saturn-pub debug`.

In [ ]:
import json
import shlex
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/saturn_pub").is_dir())
OUTPUT = ROOT / "outputs/notebooks"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Checkout:", ROOT.name)

In [ ]:
from saturn_pub.adapters.qwen import QwenAdapter
from saturn_pub.cli import Debugger
from saturn_pub.values import describe

dbg = Debugger(QwenAdapter.tiny(seed=7).session([5, 7, 11]))


def cmd(line):
    result = dbg.execute(line)
    print(f"{dbg.active}> {line}")
    if isinstance(result, dict) and "operation" in result:
        visible = {
            k: result[k]
            for k in ("operation", "boundary", "verified_exact", "fingerprint")
            if k in result
        }
    elif isinstance(result, dict) and "equal_payload" in result:
        visible = {k: result[k] for k in ("equal_payload", "left", "right")}
    else:
        visible = result
    print(json.dumps(visible, indent=2))
    return result


_ = cmd("help")

## Set a breakpoint and continue

`layer:1` means native layer 0 has completed and layer 1 is next. `step`/`next` advance one adapter transition, which may be embedding, a layer, or readout. `continue` stops at the next configured breakpoint; `continue N` always advances exactly N transitions. Breakpoint patterns accept shell-style wildcards.

In [ ]:
_ = cmd("break layer:1")
stop = cmd("continue")
assert stop["frame"]["boundary"] == "layer:1"
_ = cmd("where")
_ = cmd("capture parent")
parent_id = dbg.cuts["parent"].fingerprint

## Inspect the carrier and partial KV closure

At this stop, layer 0 has processed the pending token and later layers have not. Reads return private copies; changing a returned tensor does not mutate live execution. A typed Act is required for a live write.

In [ ]:
hidden = dbg.execute("read hidden")
print("First four hidden coordinates:", hidden[0][0][:4])
payload = dbg.session.capture(retain=False).payload
print("Per-layer KV lengths:", [key.shape[-2] for key in payload["keys"]])
_ = cmd("read tokens")

## Fork two futures and intervene

The native and candidate branches start from one unchanged cut. Zero only the candidate carrier, then execute the same remaining native model. Six transitions emit two tokens for this two-layer specimen.

In [ ]:
results = {}
for name in ("native", "candidate"):
    _ = cmd(f"fork {name} parent")
    _ = cmd(f"use {name}")
    if name == "candidate":
        _ = cmd("zero hidden")
    _ = cmd("continue 6")
    _ = cmd("read tokens")
    results[name] = describe(dbg.session.capture(retain=False).payload)
    _ = cmd("use root")
assert dbg.session.capture(retain=False).fingerprint == parent_id
_ = cmd("use native")
comparison = cmd("compare candidate")
_ = cmd("use root")

## Commit and rewind exactly

Commit adopts the candidate only when it was forked from the current parent. Restore reinstates the sealed execution closure, including each layer's KV state. Both futures remain inspectable.

In [ ]:
_ = cmd("commit candidate")
restored = cmd("restore parent")
assert restored["verified_exact"]
assert dbg.session.capture(retain=False).fingerprint == parent_id
_ = cmd("delete all")
_ = cmd("next")
_ = cmd("where")
_ = cmd("restore parent")

## Save and replay in a fresh interpreter

The cut stores execution state and weight identity; the fresh interpreter reconstructs the same random weights separately. This uses the existing public replay helper and checks the entire declared payload, including logits and KV.

In [ ]:
store = OUTPUT / "ar-debugger"
_ = cmd(f"save {shlex.quote(str(store))} parent")
fresh = subprocess.run(
    [sys.executable, str(ROOT / "examples/debugger_replay.py"), "--replay", str(store), parent_id],
    check=True,
    capture_output=True,
    text=True,
)
replayed = json.loads(fresh.stdout)
exact = {name: replayed[name]["payload"] == results[name] for name in results}
print("Fresh-process exact replay:", exact)
assert all(exact.values())

## Run the actual terminal CLI

A notebook can run a recorded CLI transcript. For an interactive prompt, open a terminal and run `saturn-pub debug --family ar`. Commands are identical. This debugger operates on supported model boundaries; it has no remote attach, Python-source breakpoints, or tensor-change watchpoints.

In [ ]:
script = OUTPUT / "commands.txt"
script.write_text("break layer:1\ncontinue\nwhere\ncapture parent\nnext\nrestore parent\nquit\n")
cli = subprocess.run(
    [sys.executable, "-m", "saturn_pub.cli", "debug", "--family", "ar", "--script", str(script)],
    check=True,
    capture_output=True,
    text=True,
)
print(cli.stdout)